# MR3a — Daily Final Join
**Inputs**: `mr3/daily_crash_mr.csv` + `mr3/address_store_mr.csv` + `mr3/type_count_store_mr.csv`
**Output**: `mr3/mr_daily_final.csv`

Join type: **inner** (zones with both crashes AND stores)
Crash-only zones remain in `daily_crash_mr.csv` for the map's blue layer.

In [4]:
import json
import pandas as pd
from collections import defaultdict

CRASH_INPUT  = 'daily_crash_mr.csv'
ADDR_INPUT   = 'address_store_mr.csv'
TYPE_INPUT   = 'type_count_store_mr.csv'
OUTPUT       = 'mr_daily_final.csv'

crashes = pd.read_csv(CRASH_INPUT)
addr    = pd.read_csv(ADDR_INPUT)
types   = pd.read_csv(TYPE_INPUT)

print(f'Crash rows : {len(crashes):,}  |  zones: {crashes["zone_id"].nunique():,}')
print(f'Store zones: {len(addr):,}')
print(f'Type rows  : {len(types):,}')

Crash rows : 2,891  |  zones: 1,490
Store zones: 1,823
Type rows  : 6,576


In [5]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# Map-side join: crash row → look up store data by zone_id

addr_lookup = addr.set_index('zone_id').to_dict('index')

type_lookup = defaultdict(dict)
for _, row in types.iterrows():
    type_lookup[row['zone_id']][row['method_of_operation']] = int(row['count'])

def mapper(crash_row):
    zid = crash_row['zone_id']
    if zid not in addr_lookup:
        return None
    store = addr_lookup[zid]
    return {
        'zone_id':           zid,
        'crash_date':        crash_row['crash_date'],
        'crashes':           crash_row['crashes'],
        'injured':           crash_row['injured'],
        'killed':            crash_row['killed'],
        'store_count':       store['store_count'],
        'active_licenses':   store['active_licenses'],
        'outdated_licenses': store['outdated_licenses'],
        'type_counts_json':  json.dumps(type_lookup.get(zid, {})),
    }

mapped = [mapper(row) for _, row in crashes.iterrows()]
mapped = [v for v in mapped if v is not None]
print(f'Mapped (inner join) rows : {len(mapped):,}')
print(f'Dropped crash-only zones : {len(crashes) - len(mapped):,}')

Mapped (inner join) rows : 2,107
Dropped crash-only zones : 784


In [6]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# Records are already fully joined — collect and save.

out = pd.DataFrame(mapped).sort_values(['zone_id','crash_date']).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Unique zones: {out["zone_id"].nunique():,}')
print(f'Saved → {OUTPUT}')

sample = out.iloc[0]
print(f'\nSample zone : {sample["zone_id"]}')
print(f'Type counts : {sample["type_counts_json"]}')
out.head()

Output rows : 2,107
Unique zones: 1,024
Saved → mr_daily_final.csv

Sample zone : 8a2a1008804ffff
Type counts : {"Tavern Serving Beer Cider Wine And Liquor": 1, "Restaurant Serving Beer, Wine, Liquor, & Cider": 1, "Restaurant Serving Beer Cider Wine And Liquor": 1}


,zone_id,crash_date,crashes,injured,killed,store_count,active_licenses,outdated_licenses,type_counts_json
0,8a2a1008804ffff,2015-11-14,1,0.0,0.0,3,0,3,"{""Tavern Serving Beer Cider Wine And Liquor"": ..."
1,8a2a1008804ffff,2018-12-04,1,1.0,0.0,3,0,3,"{""Tavern Serving Beer Cider Wine And Liquor"": ..."
2,8a2a1008814ffff,2013-06-22,1,0.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
3,8a2a1008814ffff,2017-03-11,1,0.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
4,8a2a1008814ffff,2017-05-12,1,1.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
